# Healthcare Data Governance and Cleaning

This notebook performs reproducible data-quality validation on the synthetic/de-identified patient dataset.

**Goal:** validate structure, missing values, duplicate patient IDs, ranges, categories, blood-pressure format, cholesterol, and dosage without inventing clinical values.

In [ ]:
import pandas as pd
import re

df = pd.read_csv('healthcare_patients_cleaned.csv')
df

## 1. Basic structure and data types

In [ ]:
print('Rows:', len(df))
print('Columns:', len(df.columns))

print('\nData types:')
print(df.dtypes)

## 2. Missing-value check

In [ ]:
missing = df.isnull().sum()
print(missing)
print('\nTotal missing cells:', int(missing.sum()))

## 3. Duplicate PatientID check

In [ ]:
duplicate_count = int(df['PatientID'].duplicated().sum())
print('Duplicate PatientIDs:', duplicate_count)

## 4. Range and category validation

In [ ]:
invalid_age = int(((df['Age'] < 0) | (df['Age'] > 120)).sum())
invalid_duration = int((df['TreatmentDurationDays'] <= 0).sum())
invalid_readmitted = int((~df['Readmitted'].isin(['Yes', 'No'])).sum())
invalid_cholesterol = int((df['CholesterolLevel'] < 0).sum())
invalid_dosage = int((df['DosageMg'] < 0).sum())

print('Invalid ages:', invalid_age)
print('Invalid treatment durations:', invalid_duration)
print('Invalid Readmitted values:', invalid_readmitted)
print('Invalid cholesterol values:', invalid_cholesterol)
print('Invalid dosage values:', invalid_dosage)

## 5. Blood pressure format validation

In [ ]:
bp_valid = df['BloodPressure'].astype(str).str.match(r'^\d{2,3}/\d{2,3}$')
invalid_bp = int((~bp_valid).sum())
print('Invalid blood pressure formats:', invalid_bp)

## 6. Final validation summary

The sample contains 10 records. No missing values or duplicate PatientIDs were found, and the supplied values passed the basic validation rules.

**Governance observation:** `DosageMg` should have explicit unit documentation because medication dosage conventions vary by medication. The Levothyroxine value of `0.05 mg` is retained as supplied; no clinical value was invented or converted.

In [ ]:
summary = {
    'Total records': len(df),
    'Duplicate PatientIDs': duplicate_count,
    'Missing cells': int(missing.sum()),
    'Invalid ages': invalid_age,
    'Invalid treatment durations': invalid_duration,
    'Invalid Readmitted values': invalid_readmitted,
    'Invalid blood pressure formats': invalid_bp,
    'Invalid cholesterol values': invalid_cholesterol,
    'Invalid dosage values': invalid_dosage,
}
pd.Series(summary)